# What The Values Of A PositroniumSource Mean

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

A `PositroniumSource` writes four branches into the "Hits" tree saying where
each gamma came from: `sourceType`, `decayType`, `gammaType` and `decayIndex`.
GATE stores all four as integers, and an integer says nothing about itself —
a 2 in `sourceType` is a para-positronium, a 2 in `gammaType` is an
annihilation gamma.

This notebook reads those branches from a file in which every variant of the
source occurs at once, and answers what each value means, how to ask for the
rows of one kind, and which question `decayIndex` answers — which is narrower
than it looks.

## What will you learn from this notebook

- which class describes which branch, and what its members mean
- how to read a value, or a whole column, as names
- why the classes are integers, and what goes wrong with an enum written by
  hand
- what to expect from a value the package does not know
- how to tell rows carrying decay metadata from the rest, and why that is not
  the same as "came from positronium"
- how to select rows by what a gamma was

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The example file holds 500 hits of a scene with a `PositroniumSource`
configured with every component at once, beside a back-to-back source. What it
holds is described in [`../data/README.md`](../data/README.md).

In [2]:
example_input = paths.input_example("positronium-source-all-variants.root")

In [3]:
from opengate_gate_tree import (
    DECAY_INDEX_BRANCH,
    NO_POSITRONIUM_METADATA,
    POSITRONIUM_BRANCHES,
    DecayType,
    GammaType,
    GateTree,
    SourceType,
    decode_positronium_column,
    decode_positronium_value,
    has_positronium_metadata,
    positronium_enum,
    read_tree,
)

frame = read_tree(
    example_input,
    GateTree.HITS,
    ["eventID", "edep", "sourceType", "decayType", "gammaType", DECAY_INDEX_BRANCH],
).to_dataframe()
frame

,eventID,edep,sourceType,decayType,gammaType,decayIndex
0,0,0.094538,2,1,2,0
1,0,0.416461,2,1,2,0
2,0,0.220403,2,1,2,0
3,0,0.290596,2,1,2,0
4,1,0.132729,4,1,2,1
...,...,...,...,...,...,...
495,113,0.033188,0,0,0,-1
496,113,0.225968,0,0,0,-1
497,114,0.505492,3,1,2,0
498,114,0.099978,3,1,2,0


### Three Branches, Three Classes

Three of the four branches have a class describing their values, named after
the branch it describes. `POSITRONIUM_BRANCHES` says which is which:

In [4]:
dict(POSITRONIUM_BRANCHES)

{'sourceType': <enum 'SourceType'>,
 'decayType': <enum 'DecayType'>,
 'gammaType': <enum 'GammaType'>}

`positronium_enum` answers the same question about one branch, and `None`
for a branch no class describes. `decayIndex` is one of those: it holds the
number of a component of the configured mixture, and which component a number
stands for depends on the order the fractions were configured in.

In [5]:
positronium_enum("gammaType"), positronium_enum(DECAY_INDEX_BRANCH)

(<enum 'GammaType'>, None)

The members of each class, with the integers GATE writes for them:

In [6]:
for enum_class in (SourceType, DecayType, GammaType):
    print(enum_class.__name__, {member.name: int(member) for member in enum_class})

SourceType {'NOT_DEFINED': 0, 'SINGLE_GAMMA_EMITTER': 1, 'PARA_POSITRONIUM': 2, 'ORTHO_POSITRONIUM': 3, 'DIRECT_ANNIHILATION': 4}
DecayType {'NONE': 0, 'STANDARD': 1, 'DEEXCITATION': 2}
GammaType {'UNKNOWN': 0, 'SINGLE': 1, 'ANNIHILATION': 2, 'PROMPT': 3}


### What The File Holds

Every combination of the four branches that occurs in the file, with how many
hits it covers. The values are read as names first, so that the table says what
it means:

In [7]:
named = frame.copy()
for branch in POSITRONIUM_BRANCHES:
    named[branch] = [member.name for member in decode_positronium_column(branch, frame[branch])]

named.groupby(["sourceType", "decayType", DECAY_INDEX_BRANCH, "gammaType"]).size()

sourceType           decayType     decayIndex  gammaType   
DIRECT_ANNIHILATION  STANDARD       1          ANNIHILATION     49
NOT_DEFINED          NONE          -1          UNKNOWN          61
ORTHO_POSITRONIUM    DEEXCITATION   0          ANNIHILATION     70
                                               PROMPT           47
                     STANDARD       0          ANNIHILATION    135
PARA_POSITRONIUM     DEEXCITATION   0          ANNIHILATION     38
                                               PROMPT           25
                     STANDARD       0          ANNIHILATION     75
dtype: int64

Group by group:

- 61 hits carry nothing at all — `NOT_DEFINED`, `NONE`, `UNKNOWN` and
  `decayIndex` of -1. They come from the back-to-back source, which writes no
  information about its gammas.
- Para- and ortho-positronium occur through both channels. Through the
  standard one they leave annihilation gammas only; through `DEEXCITATION` they
  leave annihilation gammas and prompt ones.
- Direct annihilation leaves annihilation gammas, through the standard channel.
- A prompt gamma occurs only where the decay went through `DEEXCITATION`, which
  is what that channel means.

Two members never occur: `SINGLE_GAMMA_EMITTER` and `SINGLE`. Both belong to
the single gamma model of the source, which this scene did not use.

### Why The Classes Are Integers

The members **are** the integers GATE wrote, so a column compares against them
as it was read. That is why the classes derive from `enum.IntEnum`, and why
they are worth importing instead of retyping. An enum written by hand for the
same purpose answers the same question with nothing — and says nothing about
it:

In [8]:
import enum


class HandWrittenGammaType(enum.Enum):
    PROMPT = 3


(
    int((frame["gammaType"] == HandWrittenGammaType.PROMPT).sum()),
    int((frame["gammaType"] == GammaType.PROMPT).sum()),
)

(0, 72)

No prompt gammas, against 72. A member of a plain `enum.Enum` is not the
integer 3, so no row equals it, and the mask is all `False` rather than an
error.

### One Value, Or A Whole Column

`decode_positronium_value` reads one value as the member it stands for:

In [9]:
decode_positronium_value("gammaType", 3)

<GammaType.PROMPT: 3>

A value the class does not know has no answer, and the error lists the values
GATE writes there:

In [10]:
try:
    decode_positronium_value("gammaType", 7)
except ValueError as error:
    print(error)

Branch 'gammaType' has no meaning for the value 7. Values GATE writes there: 0 (UNKNOWN), 1 (SINGLE), 2 (ANNIHILATION), 3 (PROMPT).


`decode_positronium_column` reads a whole column, and treats an unknown value
differently: it leaves `None` in its place and carries on, because a GATE
build can write one value more, and the rows that are understood are still
worth analysing. What it could not read is reported in the log. The package
does not configure logging on its own, so the notebook turns it on first, or
the report would go nowhere:

In [11]:
import logging

import numpy as np

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

decode_positronium_column("gammaType", np.array([2, 3, 9, 9]))

WARNING opengate_gate_tree: Branch 'gammaType' holds 1 value(s) this version does not describe: 9 (2 row(s)). They are read as nothing at all.


array([<GammaType.ANNIHILATION: 2>, <GammaType.PROMPT: 3>, None, None],
      dtype=object)

The two rows holding 9 came back as `None`, and the log says how often the
value occurred. The file itself holds no such value: every column of it
decodes in full.

In [12]:
{
    branch: sum(member is None for member in decode_positronium_column(branch, frame[branch]))
    for branch in POSITRONIUM_BRANCHES
}

{'sourceType': 0, 'decayType': 0, 'gammaType': 0}

### Which Rows Carry Decay Metadata

`decayIndex` is -1 — `NO_POSITRONIUM_METADATA` — where a row carries no decay
metadata at all. `has_positronium_metadata` gives the mask of the other rows:

In [13]:
carries_metadata = has_positronium_metadata(frame[DECAY_INDEX_BRANCH])
NO_POSITRONIUM_METADATA, int(carries_metadata.sum())

(-1, 439)

439 rows carry it. It is tempting to read that as "439 gammas came from
positronium", and the file says otherwise. Split by what each gamma was:

In [14]:
named[carries_metadata].groupby("sourceType").size()

sourceType
DIRECT_ANNIHILATION     49
ORTHO_POSITRONIUM      252
PARA_POSITRONIUM       138
dtype: int64

The direct annihilation component is part of the same `PositroniumSource`,
which numbers its gammas like the rest, so its 49 hits carry metadata too —
and form no positronium. "Carries metadata" is what the mask guarantees; what a
gamma was is said by `sourceType`. The gammas of positronium are the ones of
its two members:

In [15]:
from_positronium = frame["sourceType"].gate.is_source_type(
    SourceType.PARA_POSITRONIUM, SourceType.ORTHO_POSITRONIUM
)
int(from_positronium.sum())

390

390 = 439 − 49. The frame answers the metadata question directly as well,
through the `gate` namespace, with the rows or with the mask:

In [16]:
len(frame.gate.with_decay_metadata()), int(frame.gate.has_decay_metadata().sum())

(439, 439)

### Selecting By What A Gamma Was

Each branch has a selector that takes the members of its own class, as many as
the question needs. `is_*` answers with the mask that selects rows:

In [17]:
prompt_or_annihilation = frame["gammaType"].gate.is_gamma_type(
    GammaType.PROMPT, GammaType.ANNIHILATION
)
int(prompt_or_annihilation.sum())

439

`select_by_*` answers with the values of the column itself — a `decayType`
column holding nothing but `DEEXCITATION`, the integer 2, which is what a count is made
from:

In [18]:
frame["decayType"].gate.select_by_decay_type(DecayType.DEEXCITATION).value_counts()

decayType
2    180
Name: count, dtype: int64

The classes share their numbers, so a member of the wrong class would select
by a value meaning something else. It is refused instead:

In [19]:
try:
    frame["gammaType"].gate.is_gamma_type(SourceType.PARA_POSITRONIUM)
except ValueError as error:
    print(error)

Selecting by GammaType takes its own members, got SourceType.PARA_POSITRONIUM. The classes share their numbers, so another class would select by a value that means something else.


### What A Prompt Gamma Leaves

The names make a question about the physics read as one. The energy each kind
of gamma deposited, in MeV, as GATE writes it:

In [20]:
named.groupby("gammaType")["edep"].agg(["count", "mean", "max"])

,count,mean,max
gammaType,,,
ANNIHILATION,367,0.269518,0.510999
PROMPT,72,0.541667,1.300000
UNKNOWN,61,0.301574,0.511000


No annihilation or unknown gamma deposited more than 511 keV, which is all an
annihilation gamma carries; the prompt gammas reach 1.3 MeV. A file written by
the package keeps the integers rather than the names: the names are a reading
of the data, and a branch of text could not be read back as the whole numbers
every structure of the tree describes.

## Summary

The values a `PositroniumSource` writes are read through `SourceType`,
`DecayType` and `GammaType`, whose members are the integers themselves, so a
column compares against them as it was read; `decode_positronium_value` and
`decode_positronium_column` read them as names. `decayIndex` tells rows
carrying decay metadata from the rest — which includes direct annihilation —
and `sourceType` says what a gamma actually was.

## Bibliography

- [PositroniumSource Data](https://opengate-gate-tree.readthedocs.io/en/latest/guide/positronium.html)
  — the classes, their members, and what `decayIndex` guarantees
- [Filtering And Selecting](https://opengate-gate-tree.readthedocs.io/en/latest/guide/filtering.html)
  — the selectors of the `gate` namespace
- [Using the package as a library](https://opengate-gate-tree.readthedocs.io/en/latest/guide/library.html)
  — why the package configures no logging of its own
- `GateEmittedGammaInformation.hh` in the GATE sources — the enums the classes
  are taken from

## Changelog

- v1: written.